In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content/drive/MyDrive/VirconPaper1

In [ ]:
# ============================================================
# VIRCON — FINAL HUMAN AUDIT FREEZE
# Two-rater agreement + adjudicated 200-message reference
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.metrics import (
    cohen_kappa_score,
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

WORKDIR = Path("/content/drive/MyDrive/VirconPaper1")

CANDIDATES = [
    WORKDIR / "triage_manual_audit_two_rater_completed.csv",
    WORKDIR / "triage_manual_audit_two_rater_completed(1).csv",
    WORKDIR / "triage_manual_audit_two_rater_completed(2).csv",
]

# Prefer the most recently modified available file
existing = [p for p in CANDIDATES if p.is_file()]

if not existing:
    raise FileNotFoundError(
        "Could not find the completed audit file in VirconPaper1."
    )

AUDIT_PATH = max(existing, key=lambda p: p.stat().st_mtime)

print("Using:")
print(AUDIT_PATH)

audit = pd.read_csv(AUDIT_PATH).fillna("")

# ------------------------------------------------------------
# 2. Validate structure
# ------------------------------------------------------------

required = [
    "global_row_id",
    "label_triage",
    "rater1_triage",
    "rater2_triage",
    "adjudicated_triage",
]

missing = [c for c in required if c not in audit.columns]

if missing:
    raise KeyError(
        f"Missing required columns: {missing}\n"
        f"Available columns: {list(audit.columns)}"
    )

VALID = {"request", "response", "other", "uncertain"}

for col in [
    "label_triage",
    "rater1_triage",
    "rater2_triage",
    "adjudicated_triage",
]:
    audit[col] = (
        audit[col]
        .astype(str)
        .str.strip()
        .str.lower()
    )

for col in [
    "rater1_triage",
    "rater2_triage",
    "adjudicated_triage",
]:
    bad = ~audit[col].isin(VALID)

    if bad.any():
        print("\nInvalid rows in:", col)
        display(
            audit.loc[
                bad,
                [
                    c for c in [
                        "global_row_id",
                        "raw_text",
                        col,
                    ]
                    if c in audit.columns
                ]
            ]
        )
        raise ValueError(
            f"{col} contains invalid or blank labels."
        )

if len(audit) != 200:
    raise RuntimeError(
        f"Expected 200 audited rows, found {len(audit)}."
    )

if audit["global_row_id"].duplicated().any():
    raise RuntimeError(
        "Duplicate global_row_id values found in audit."
    )

# ------------------------------------------------------------
# 3. Preserve original independent rater agreement
# ------------------------------------------------------------

audit["original_rater_agreement"] = (
    audit["rater1_triage"] ==
    audit["rater2_triage"]
)

n_agreed = int(
    audit["original_rater_agreement"].sum()
)

n_disagreed = int(
    (~audit["original_rater_agreement"]).sum()
)

raw_agreement = float(
    audit["original_rater_agreement"].mean()
)

kappa = float(
    cohen_kappa_score(
        audit["rater1_triage"],
        audit["rater2_triage"],
        labels=[
            "request",
            "response",
            "other",
            "uncertain",
        ],
    )
)

print("\nOriginal independent-rater agreement")
print("------------------------------------")
print("N audited:", len(audit))
print("Agreed:", n_agreed)
print("Disagreed:", n_disagreed)
print("Raw agreement:", raw_agreement)
print("Cohen kappa:", kappa)

# ------------------------------------------------------------
# 4. Create final adjudicated human label
# ------------------------------------------------------------

audit["final_human_triage"] = (
    audit["adjudicated_triage"]
)

final_counts = (
    audit["final_human_triage"]
    .value_counts()
    .reindex(
        ["request", "response", "other", "uncertain"],
        fill_value=0
    )
)

final_percent = (
    final_counts / len(audit) * 100
)

print("\nFinal adjudicated distribution")
print("------------------------------")

for label in final_counts.index:
    print(
        f"{label:10s}: "
        f"{final_counts[label]:3d} "
        f"({final_percent[label]:.2f}%)"
    )

# ------------------------------------------------------------
# 5. Hidden actionability among taxonomy-derived OTHER
# ------------------------------------------------------------

proxy_other = audit[
    audit["label_triage"].eq("other")
].copy()

hidden_request = (
    proxy_other["final_human_triage"]
    .eq("request")
)

hidden_response = (
    proxy_other["final_human_triage"]
    .eq("response")
)

hidden_actionable = (
    hidden_request |
    hidden_response
)

hidden_request_n = int(hidden_request.sum())
hidden_response_n = int(hidden_response.sum())
hidden_actionable_n = int(hidden_actionable.sum())

hidden_actionable_rate = float(
    hidden_actionable.mean()
)

print("\nHidden actionability within taxonomy-'other'")
print("-------------------------------------------")
print("Audited taxonomy-other:", len(proxy_other))
print("Hidden requests:", hidden_request_n)
print("Hidden responses:", hidden_response_n)
print("Hidden actionable total:", hidden_actionable_n)
print(
    "Hidden actionable rate:",
    f"{100*hidden_actionable_rate:.2f}%"
)

# ------------------------------------------------------------
# 6. Bootstrap CI for hidden actionable fraction
# ------------------------------------------------------------

RNG = np.random.default_rng(20261003)
N_BOOT = 10000

values = hidden_actionable.to_numpy(dtype=float)

boot = np.empty(N_BOOT)

for b in range(N_BOOT):
    idx = RNG.integers(
        0,
        len(values),
        len(values)
    )
    boot[b] = values[idx].mean()

ci_low, ci_high = np.percentile(
    boot,
    [2.5, 97.5]
)

print(
    "95% bootstrap CI:",
    f"{ci_low:.4f} to {ci_high:.4f}"
)

# ------------------------------------------------------------
# 7. Compare taxonomy triage against final human reference
# ------------------------------------------------------------

determinate = audit[
    audit["final_human_triage"]
    .isin(["request", "response", "other"])
].copy()

proxy_accuracy = float(
    accuracy_score(
        determinate["final_human_triage"],
        determinate["label_triage"],
    )
)

proxy_macro_f1 = float(
    f1_score(
        determinate["final_human_triage"],
        determinate["label_triage"],
        labels=["request", "response", "other"],
        average="macro",
        zero_division=0,
    )
)

print("\nTaxonomy proxy vs final human audit")
print("-----------------------------------")
print("Accuracy:", proxy_accuracy)
print("Macro-F1:", proxy_macro_f1)

report = classification_report(
    determinate["final_human_triage"],
    determinate["label_triage"],
    labels=["request", "response", "other"],
    output_dict=True,
    zero_division=0,
)

report_df = (
    pd.DataFrame(report)
    .T
    .reset_index()
    .rename(columns={"index": "label"})
)

# ------------------------------------------------------------
# 8. Final confusion matrix
# ------------------------------------------------------------

cm = confusion_matrix(
    determinate["final_human_triage"],
    determinate["label_triage"],
    labels=["request", "response", "other"],
)

cm_df = pd.DataFrame(
    cm,
    index=[
        "human_request",
        "human_response",
        "human_other",
    ],
    columns=[
        "proxy_request",
        "proxy_response",
        "proxy_other",
    ],
)

# ------------------------------------------------------------
# 9. Per-original-humanitarian-class audit
# ------------------------------------------------------------

if "label_11" in audit.columns:

    per_class = (
        audit
        .groupby(
            [
                "label_11",
                "final_human_triage"
            ]
        )
        .size()
        .rename("n")
        .reset_index()
    )

    totals = (
        per_class
        .groupby("label_11")["n"]
        .transform("sum")
    )

    per_class["fraction"] = (
        per_class["n"] / totals
    )

else:
    per_class = pd.DataFrame()

# ------------------------------------------------------------
# 10. Adjudication-specific record
# ------------------------------------------------------------

adjudication_rows = audit[
    ~audit["original_rater_agreement"]
].copy()

adjudication_export_cols = [
    c for c in [
        "global_row_id",
        "source_split",
        "source_row",
        "event",
        "label_11",
        "label_triage",
        "rater1_triage",
        "rater2_triage",
        "adjudicated_triage",
        "final_human_triage",
    ]
    if c in adjudication_rows.columns
]

adjudication_rows = (
    adjudication_rows[
        adjudication_export_cols
    ]
)

# ------------------------------------------------------------
# 11. Reviewer-facing summary
# ------------------------------------------------------------

summary = pd.DataFrame(
    [
        {
            "n_audited":
                int(len(audit)),

            "original_rater_agreement_n":
                n_agreed,

            "original_rater_disagreement_n":
                n_disagreed,

            "raw_interrater_agreement":
                raw_agreement,

            "cohen_kappa":
                kappa,

            "n_adjudicated":
                n_disagreed,

            "final_request_count":
                int(final_counts["request"]),

            "final_response_count":
                int(final_counts["response"]),

            "final_other_count":
                int(final_counts["other"]),

            "final_uncertain_count":
                int(final_counts["uncertain"]),

            "hidden_request_count":
                hidden_request_n,

            "hidden_response_count":
                hidden_response_n,

            "hidden_actionable_count":
                hidden_actionable_n,

            "hidden_actionable_fraction":
                hidden_actionable_rate,

            "hidden_actionable_ci95_low":
                float(ci_low),

            "hidden_actionable_ci95_high":
                float(ci_high),

            "taxonomy_proxy_accuracy":
                proxy_accuracy,

            "taxonomy_proxy_macro_f1":
                proxy_macro_f1,

            "bootstrap_resamples":
                N_BOOT,
        }
    ]
)

# ------------------------------------------------------------
# 12. Save final frozen outputs
# ------------------------------------------------------------

FINAL_DIR = (
    WORKDIR /
    "VIRCON_FINAL_HUMAN_AUDIT"
)

FINAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

audit.to_csv(
    FINAL_DIR /
    "triage_manual_audit_FINAL_adjudicated_200.csv",
    index=False,
)

summary.to_csv(
    FINAL_DIR /
    "triage_manual_audit_FINAL_frozen_summary.csv",
    index=False,
)

report_df.to_csv(
    FINAL_DIR /
    "triage_manual_audit_FINAL_proxy_classification_report.csv",
    index=False,
)

cm_df.to_csv(
    FINAL_DIR /
    "triage_manual_audit_FINAL_confusion_matrix.csv"
)

adjudication_rows.to_csv(
    FINAL_DIR /
    "triage_manual_audit_FINAL_7_adjudicated_cases.csv",
    index=False,
)

if not per_class.empty:
    per_class.to_csv(
        FINAL_DIR /
        "triage_manual_audit_FINAL_per_source_class.csv",
        index=False,
    )

# Public-safe copy without raw tweet text
safe_cols = [
    c for c in [
        "global_row_id",
        "source_split",
        "source_row",
        "event",
        "label_11",
        "label_triage",
        "rater1_triage",
        "rater2_triage",
        "adjudicated_triage",
        "final_human_triage",
        "original_rater_agreement",
    ]
    if c in audit.columns
]

audit[safe_cols].to_csv(
    FINAL_DIR /
    "triage_manual_audit_FINAL_public_safe.csv",
    index=False,
)

print("\n========================================")
print("FINAL HUMAN AUDIT FROZEN")
print("========================================")
print("Folder:")
print(FINAL_DIR)

print("\nReviewer-facing summary:")
display(summary)

print("\nConfusion matrix:")
display(cm_df)

if not per_class.empty:
    print("\nPer-source-class distribution:")
    display(per_class)